# Practical 6 – Self-Attention, Scaled Dot-Product Attention and Multi-Head Attention

## Objective
Implement self-attention, scaled dot-product attention, and multi-head attention used in Transformer networks.

## Requirements
Python, PyTorch

## 1. Import libraries

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

## 2. Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1))
    scores = scores / math.sqrt(d_k)

    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)

    return output, weights

Q = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
K = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
V = torch.tensor([[[10.0, 0.0], [0.0, 20.0]]])

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights:\n", weights)
print("\nAttention output:\n", output)

## 3. Why scaling is used

In [ ]:
d_k = Q.size(-1)
raw_scores = torch.matmul(Q, K.transpose(-2, -1))

print("Raw dot-product scores:")
print(raw_scores)

print("\nScaled scores:")
print(raw_scores / math.sqrt(d_k))

print("\nScaling by sqrt(d_k) prevents very large values before softmax.")

## 4. Multi-Head Attention from scratch

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=8, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch, seq_len, _ = x.shape
        x = x.view(batch, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch, heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch, seq_len, heads * head_dim)

    def forward(self, x):
        Q = self.split_heads(self.Wq(x))
        K = self.split_heads(self.Wk(x))
        V = self.split_heads(self.Wv(x))

        attention_output, attention_weights = scaled_dot_product_attention(Q, K, V)
        combined = self.combine_heads(attention_output)

        return self.out(combined), attention_weights

mha = MultiHeadAttention(d_model=8, num_heads=2)

x = torch.randn(1, 4, 8)
mha_output, mha_weights = mha(x)

print("Input shape :", x.shape)
print("Output shape:", mha_output.shape)
print("Attention weight shape:", mha_weights.shape)

## 5. Inspect the two attention heads

In [ ]:
for head in range(2):
    print(f"\nHead {head+1} attention:")
    print(mha_weights[0, head])

## Result
Scaled dot-product attention computes attention using Q, K and V. Multi-head attention performs this operation in multiple representation subspaces and combines the results.